# Custom Crystal Dataset (PyTorch)

The `CrystalDataset` class handles periodic crystal structures using pymatgen. It reads CIF files,
serializes structures to JSON, and generates graph representations with periodic boundary conditions.

This tutorial demonstrates:
1. Creating a custom `CrystalDataset` subclass
2. Crystal graph construction from pymatgen structures
3. Periodic boundary handling with `set_range_periodic`
4. Converting to PyG Data objects
5. Training CGCNN or SchNet on crystal data

## 0. Create Example Data

We create a few simple crystal structures using pymatgen and write them as CIF files.

In [ ]:
import numpy as np
import os
import pymatgen
import pymatgen.core.structure

test_data = [
    pymatgen.core.Structure(
        lattice=np.array([
            [4.34157255, 0., 2.50660808],
            [1.44719085, 4.09327385, 2.50660808],
            [0., 0., 5.01321616]
        ]),
        species=["Te", "Ba"],
        coords=np.array([[0.5, 0.5, 0.5], [0., 0., 0.]])
    ),
    pymatgen.core.Structure(
        lattice=np.array([
            [2.95117784, 0., 1.70386332],
            [0.98372595, 2.78239715, 1.70386332],
            [0., 0., 3.40772664]
        ]),
        species=["B", "As"],
        coords=np.array([[0.25, 0.25, 0.25], [0., 0., 0.]])
    ),
    pymatgen.core.Structure(
        lattice=np.array([
            [4.3015, 0., 0.],
            [-2.15075, 3.725208, 0.],
            [0., 0., 5.2703]
        ]),
        species=["Ba", "Ga", "Si", "H"],
        coords=np.array([
            [0., 0., 0.],
            [0.6666, 0.3333, 0.5423],
            [0.3334, 0.6667, 0.4555],
            [0.6666, 0.3333, 0.8759]
        ])
    ),
]

os.makedirs("ExampleCrystal", exist_ok=True)
os.makedirs("ExampleCrystal/CifFiles", exist_ok=True)

for i, x in enumerate(test_data):
    x.to(filename=f"ExampleCrystal/CifFiles/file_{i}.cif", fmt="cif")

csv_data = "\n".join([
    "file_name,index,label",
    "file_0.cif, 0, 98.58577122703691",
    "file_1.cif, 1, 701.5857233477558",
    "file_2.cif, 2, 1138.5856886491724"
])
with open("ExampleCrystal/data.csv", "w") as f:
    f.write(csv_data)

print("Crystal data created.")

Expected file structure:

```
ExampleCrystal/
    CifFiles/
        file_0.cif
        file_1.cif
        file_2.cif
    data.csv
    data.pymatgen.json   # Created by prepare_data()
```

## 1. Initialization

In [ ]:
from kgcnn_torch.data.crystal import CrystalDataset

dataset = CrystalDataset(
    data_directory="ExampleCrystal/",
    dataset_name="ExampleCrystal",
    file_name="data.csv",
    file_directory="CifFiles"
)

## 2. Prepare Data

`prepare_data()` reads individual CIF files from the file directory and serializes all pymatgen
structures into a single JSON file for faster subsequent loading.

In [ ]:
dataset.prepare_data(file_column_name="file_name", overwrite=True)

## 3. Read into Memory

`read_in_memory()` loads the serialized pymatgen structures and extracts
node coordinates, fractional coordinates, lattice matrix, atomic numbers, and labels.

In [ ]:
dataset.read_in_memory(label_column_name="label")

print(f"Number of crystals: {len(dataset)}")
print(f"Properties: {list(dataset[0].keys())}")
print(f"\nGraph 0: {dataset[0]}")

### Accessing Pymatgen Structures Directly

You can also retrieve the raw pymatgen `Structure` objects from the JSON file.

In [ ]:
structs = dataset.get_structures_from_json_file()
for s in structs:
    print(f"  {s.composition}: {len(s)} sites, volume={s.lattice.volume:.2f}")

## 4. Build Periodic Graph

Crystal graphs require periodic boundary handling. The `set_range_periodic` preprocessor
finds neighbours across periodic images and returns:
- `range_indices`: (i, j) pairs
- `range_image`: lattice translation vectors for periodic images
- `range_attributes`: pairwise distances

In [ ]:
dataset.map_list(
    method="set_range_periodic",
    max_distance=5.0,
    max_neighbours=20
)

print("Graph 0 after periodic range construction:")
print(f"  range_indices shape: {dataset[0]['range_indices'].shape}")
print(f"  range_image shape: {dataset[0]['range_image'].shape}")
print(f"  range_attributes shape: {dataset[0]['range_attributes'].shape}")
print(f"  Sample distances: {dataset[0]['range_attributes'][:5].flatten()}")

In [ ]:
# Full graph dictionary for crystal 0
print(dataset[0])

## 5. Convert to PyG Data and Train

`to_pyg_list()` automatically handles crystal-specific attributes like `lattice` and `edge_image`
(periodic image vectors). These are passed to the model for computing distances with periodic shifts.

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader

pyg_list = dataset.to_pyg_list(
    node_key="node_number",
    pos_key="node_coordinates",
    edge_key="range_indices",
    label_key="graph_labels",
    lattice_key="graph_lattice",
    image_key="range_image"
)

print(f"Number of PyG graphs: {len(pyg_list)}")
print(f"Example graph: {pyg_list[0]}")
print(f"  z: {pyg_list[0].z}")
print(f"  lattice: {pyg_list[0].lattice.shape}")
print(f"  edge_image: {pyg_list[0].edge_image.shape}")

### Train CGCNN

CGCNN is specifically designed for crystal property prediction. It uses Gaussian-expanded
distances as edge features and gated convolution layers.

In [ ]:
from kgcnn_torch.models.cgcnn import CGCNNModel

cgcnn = CGCNNModel(
    node_dim=64,
    depth=3,
    gauss_bins=40,
    gauss_distance=5.0,
    gauss_sigma=0.4,
    conv_activation="softplus",
    node_pooling="mean",
    output_units=[64, 32],
    output_activation="softplus",
    num_targets=1,
    make_distance=False,    # We precomputed distances
    expand_distance=True    # Gaussian expansion in the model
)

print(cgcnn)

In [ ]:
# Prepare edge_attr from range_attributes (distances) for CGCNN
# CGCNN expects edge_attr; to_pyg_list does not auto-set it from range,
# so we manually assign.
for data in pyg_list:
    # range_attributes (distances) were stored - get from original dataset
    pass  # edge_attr will be computed by model's Gaussian expansion

# For CGCNN with make_distance=False, we need to provide edge_attr as distances.
# Let us recompute from the range_attributes.
for i, data in enumerate(pyg_list):
    dist = dataset[i].obtain_property("range_attributes")
    if dist is not None:
        data.edge_attr = torch.tensor(np.asarray(dist), dtype=torch.float)

In [ ]:
from kgcnn_torch.training.trainer import fit

# Scale labels for training
y_values = np.array([float(d.y.item()) for d in pyg_list])
y_mean, y_std = y_values.mean(), y_values.std()
for d in pyg_list:
    d.y = (d.y - y_mean) / max(y_std, 1e-7)

train_loader = DataLoader(pyg_list, batch_size=3, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

history = fit(
    model=cgcnn,
    train_loader=train_loader,
    val_loader=train_loader,
    optimizer=torch.optim.Adam(cgcnn.parameters(), lr=1e-3),
    loss_fn=nn.MSELoss(),
    epochs=30,
    device=device,
    metrics={"mae": lambda pred, target: torch.mean(torch.abs(pred - target))},
    verbose=1
)

print(f"Final train loss: {history['train_loss'][-1]:.4f}")

### Alternative: Train SchNet for Crystals

The `SchNetCrystalModel` variant handles periodic boundary conditions by using
`lattice` and `edge_image` to compute distances with periodic shifts.

In [ ]:
from kgcnn_torch.models.schnet import SchNetCrystalModel

schnet_crystal = SchNetCrystalModel(
    node_dim=64,
    depth=4,
    units=128,
    gauss_bins=25,
    gauss_distance=5.0,
    gauss_sigma=0.4,
    node_pooling="mean",
    last_mlp_units=[128, 64],
    num_targets=1,
    make_distance=True,      # Compute distances from positions + lattice
    expand_distance=True
)

# Reload fresh labels (re-scale)
for i, data in enumerate(pyg_list):
    data.y = torch.tensor([(y_values[i] - y_mean) / max(y_std, 1e-7)], dtype=torch.float)

train_loader = DataLoader(pyg_list, batch_size=3, shuffle=True)

history = fit(
    model=schnet_crystal,
    train_loader=train_loader,
    val_loader=train_loader,
    optimizer=torch.optim.Adam(schnet_crystal.parameters(), lr=1e-4),
    loss_fn=nn.MSELoss(),
    epochs=30,
    device=device,
    verbose=1
)

print(f"Final train loss: {history['train_loss'][-1]:.4f}")

## Summary

This notebook demonstrated the full workflow for using `CrystalDataset` in kgcnn-torch:

1. **Create CIF files** -- Store crystal structures as CIF
2. **prepare_data()** -- Parse CIF files into serialized pymatgen JSON
3. **read_in_memory()** -- Load coordinates, lattice, atomic numbers, labels
4. **map_list("set_range_periodic")** -- Build periodic crystal graph
5. **to_pyg_list()** -- Convert to PyG Data (with lattice and edge_image)
6. **fit()** -- Train CGCNN or SchNetCrystalModel with PyTorch